# Call Foundry Agent via Responses API
This notebook calls an **Azure AI Foundry Agent** (`Databricks-F1-SPN`) using the `azure-ai-projects` SDK and the OpenAI Responses API. Authentication uses a service principal whose credentials are stored in an Azure Key Vault-backed secret scope (`DatabricksKV-JackBender`). The agent connects to a **Databricks Genie space** via an MCP tool to answer questions.

**Setup:**
1. Install the `azure-ai-projects` SDK
2. Restart the Python environment
3. Authenticate with `ClientSecretCredential`, obtain an OpenAI client bound to the agent, and call `responses.create()`

> The service principal requires the **Cognitive Services OpenAI User** RBAC role. On first run, an OAuth consent link may be returned — visit it to authorize the Foundry agent's connection to Databricks, then re-run.

> Also you must create a Databricks Custom App in the Account Console and use these credentials in your MCP tool in AI Foundry to connect to your Databricks Genie Space from AI Foundry.

In [0]:
%pip install azure-ai-projects>=2.1.0 --quiet

In [0]:
dbutils.library.restartPython() 

In [0]:
from azure.identity import ClientSecretCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import VersionRefIndicator

key_vault_name = "DatabricksKV-JackBender"

# Authenticate using service principal credentials from Key Vault
credential = ClientSecretCredential(
    tenant_id=dbutils.secrets.get(scope=key_vault_name, key="SPTenantID"),
    client_id=dbutils.secrets.get(scope=key_vault_name, key="SPDatabricksAppID"),
    client_secret=dbutils.secrets.get(scope=key_vault_name, key="SPDatabricks"),
)

endpoint="https://<project name>-resource.services.ai.azure.com/api/projects/<project name>",  # Foundry API Endpoint on the Foundry Resource
agent_name = "<agent name>"  #Foundry Agent Name
with AIProjectClient(
    endpoint=endpoint,
    credential=credential,
    allow_preview=True,
) as project_client:
    # Get an OpenAI client bound to the agent endpoint (no session needed for single-turn)
    openai_client = project_client.get_openai_client(agent_name=agent_name)

    # Call Responses API directly without session state
    response = openai_client.responses.create(
        input="Who won the Canadian Grand Prix in 2026?",
    )

    # Check for OAuth consent requests before displaying output
    consent_link = None
    for item in response.output:
        if item.type == "oauth_consent_request":
            consent_link = getattr(item, 'model_extra', {}).get('consent_link')
            break

    if consent_link:
        print("⚠️ OAuth consent required. Visit this link to authorize the Foundry agent's connection to Databricks:")
        print(f"\n{consent_link}")
        print("\nAfter granting consent, re-run this cell.")
    elif response.output_text:
        display(response.output_text)
    else:
        print("No text output received. Full response output:")
        for i, item in enumerate(response.output):
            print(f"  [{i}] type={item.type}, extra={getattr(item, 'model_extra', {})}")